In [0]:
# Databricks notebook source
# MAGIC %md
# MAGIC # 03 — Gold Transactions
# MAGIC **Layer:** Gold | Reconciliation, business enrichment, partitioned output
# MAGIC
# MAGIC - LEFT JOIN: core_banking (authoritative) vs card_auth
# MAGIC - Composite key: `card_last_four + amount + transaction_date`
# MAGIC - Reconciliation statuses: MATCHED / MISSING_AUTH / NO_AUTH_EXPECTED / AMOUNT_MISMATCH
# MAGIC - Partitioned by `transaction_date`

# COMMAND ----------

from pyspark.sql.functions import (
    col, abs as spark_abs, when, current_timestamp,
    lit, to_date, round as spark_round
)
from pyspark.sql.types import DecimalType

CATALOG     = "fintech_lakehouse_dev"
SCHEMA      = "transactions"
SILVER_CORE = f"{CATALOG}.{SCHEMA}.silver_core_banking"
SILVER_CARD = f"{CATALOG}.{SCHEMA}.silver_card_auth"
GOLD_TABLE  = f"{CATALOG}.{SCHEMA}.gold_transactions"

In [0]:
# COMMAND ----------
# Load Silver (DQ-passed records only)
core_df = spark.table(SILVER_CORE)
card_df = spark.table(SILVER_CARD)

print(f"Silver core (dq_passed): {core_df.count()}")
print(f"Silver card (dq_passed): {card_df.count()}")

In [0]:
# COMMAND ----------
# Composite key join — card_last_four + amount + transaction_date
# transaction_id join not viable (independently generated datasets)

card_select = card_df.select(
    col("transaction_id").alias("auth_txn_id"),
    col("amount").cast(DecimalType(18,2)).alias("auth_amount"),
    col("authorization_code"),
    col("status").alias("auth_status"),
    col("card_last_four").alias("auth_card_last_four"),
    to_date(col("transaction_timestamp")).alias("auth_date")
)

joined = core_df.join(
    card_select,
    (core_df["card_last_four"] == card_select["auth_card_last_four"]) &
    (core_df["amount"].cast(DecimalType(18,2)) == card_select["auth_amount"]) &
    (to_date(core_df["transaction_timestamp"]) == card_select["auth_date"]),
    how="left"
).drop("auth_card_last_four", "auth_date", "auth_txn_id")

gold_df = (
    joined
    .withColumn("amount_variance",
        spark_round(spark_abs(col("amount") - col("auth_amount")), 2))
    .withColumn("reconciliation_status",
        when(col("auth_amount").isNull() & (col("transaction_type") == "PURCHASE"),
             lit("MISSING_AUTH"))
        .when(col("auth_amount").isNull(),
             lit("NO_AUTH_EXPECTED"))
        .when(col("amount_variance") > 0.01,
             lit("AMOUNT_MISMATCH"))
        .otherwise(lit("MATCHED"))
    )
    .withColumn("is_high_value", col("amount") > 500)
    .withColumn("transaction_date", to_date(col("transaction_timestamp")))
    .withColumn("gold_processed_at", current_timestamp())
    .drop("silver_processed_at", "silver_source_table",
          "dq_passed", "dq_amount_invalid", "dq_timestamp_invalid",
          "dq_currency_invalid", "dq_txn_type_invalid",
          "dq_status_invalid", "dq_card_format_invalid")
)

In [0]:
# COMMAND ----------
# Reconciliation preview before write
print("Reconciliation status breakdown:")
gold_df.groupBy("reconciliation_status", "transaction_type") \
       .count() \
       .orderBy("reconciliation_status", "count") \
       .show()

In [0]:
# COMMAND ----------
# Write Gold table — partitioned by transaction_date
gold_df.write.format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .partitionBy("transaction_date") \
    .saveAsTable(GOLD_TABLE)

row_count = spark.table(GOLD_TABLE).count()
print(f"Gold table row count: {row_count}")
print(f"Written → {GOLD_TABLE}")

In [0]:
%sql
-- COMMAND ----------
%sql
-- Gold layer verification
SELECT
    reconciliation_status,
    transaction_type,
    COUNT(*) AS total_txns,
    ROUND(SUM(amount), 2) AS total_settled_amount,
    ROUND(AVG(amount_variance), 4) AS avg_variance,
    SUM(CASE WHEN is_high_value THEN 1 ELSE 0 END) AS high_value_count
FROM fintech_lakehouse_dev.transactions.gold_transactions
GROUP BY reconciliation_status, transaction_type
ORDER BY reconciliation_status, total_txns DESC